# CASMI26 - fingerprint networks, held-out retrain (warm-up job)

Trains the per-spectrum and the merged spectrum->fingerprint networks on a T4 and writes
`/kaggle/working/out/export/` (`fp_single_ho1.pt`, `fp_merged_ho1.pt`, `train_result.json`, logs).

Settings: **Accelerator = GPU T4 x2 (or T4)**, Internet off is fine, private dataset `casmi-train-pkg` attached.
Use **Save Version -> Save & Run All (Commit)** so the run survives closing the browser.
If a session ends before it is finished: add this notebook's own output as an input and run it again - it resumes.

In [ ]:
import glob, os, subprocess, sys
hits = sorted(glob.glob('/kaggle/input/**/train_fp.py', recursive=True))
assert hits, 'attach the private dataset casmi-train-pkg (Add Input)'
PKG = os.path.dirname(hits[0]); print('package:', PKG)
prev = sorted(set(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)))
print('earlier checkpoints:', prev or 'none')
!nvidia-smi -L
import torch; print(torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'turn on the GPU: Settings -> Accelerator -> GPU T4'

In [ ]:
# 660 minutes keeps the run inside the 12-hour session limit; a second run resumes automatically.
cmd = [sys.executable, f'{PKG}/train_fp.py', '--data', PKG, '--out', '/kaggle/working/out', '--max_minutes', '660']
if prev: cmd += ['--resume_from', prev[0]]
rc = subprocess.call(cmd); print('exit code', rc, '(0 = finished, 2 = not finished yet, run again)')

In [ ]:
rc2 = subprocess.call([sys.executable, f'{PKG}/check_result.py', '--data', PKG, '--out', '/kaggle/working/out'])
print(open('/kaggle/working/out/export/check_summary.txt').read())
!ls -la /kaggle/working/out /kaggle/working/out/export